## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

# Three Parts to this lab

## Part 1: A simple "Agent" and "Agent Loop"

Basically an LLM call. We'll add tracing and streaming to the mix.

## Part 2: Adding a Tool

A familiar one, but oh-so-easy

## Part 3: Adding Memory

So that different Agent calls know about each other

In [27]:
# The imports

import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession

load_dotenv(override=True)


True

## Sidenote

The actual name of this framework on the official Python index pypi.org is `openai-agents`

So for your own projects in the future, you would do:

`pip install openai-agents`  
or  
`uv add openai-agents`

followed by

`from agents import Agent, Runner, trace`

Beware that doing a `pip install agents` would install something completely different - an older reinforcement learning library.


In [28]:
# Calling an agent with a trace

agent = Agent(name="Jokester", instructions = "You are a joke teller", model="gpt-4o-mini")
with trace("Telling a joke"):
    Result = await Runner.run(agent, "Tell me a joke about people from Kolkata")
print (Result.final_output)

## The following gives a list of both the user message and the assistant response
import textwrap
print (textwrap.fill(str(Result.to_input_list()), width=120))

Why did the Kolkatan bring a ladder to the bakery?

Because they heard the pastries were on another level!
[{'content': 'Tell me a joke about people from Kolkata', 'role': 'user'}, {'id':
'msg_091123a1a26e9e0f006a95686f828c87d1bf79c892c62e6ce0', 'content': [{'annotations': [], 'text': 'Why did the Kolkatan
bring a ladder to the bakery?\n\nBecause they heard the pastries were on another level!', 'type': 'output_text',
'logprobs': []}], 'role': 'assistant', 'status': 'completed', 'type': 'message'}]


## Now go and look at the trace

https://platform.openai.com/traces

In [4]:
# Streaming
Result = Runner.run_streamed(agent,"Please tell me 5 jokes about people from Kolkata")
async for event in Result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print (event.data.delta, end="", flush=True)

Sure! Here are five light-hearted jokes inspired by the vibrant culture of Kolkata:

1. **Why do people from Kolkata carry a football around?**  
   Because they can't resist a chance to *pass* the time!

2. **How do you know someone is from Kolkata?**  
   Don't worry; they'll tell you… right after they finish their *rossogolla*!

3. **What did the Kolkatan say when he saw a sunburnt tourist?**  
   "You must've mistaken this for the *maidan*! You can't just *burn* here!"

4. **Why do Kolkatans make great detectives?**  
   Because they can *spot* a good *kathi roll* from a mile away!

5. **Why did the fish refuse to leave the rivers of Kolkata?**  
   Because it couldn't resist the *fish fry* parties!

Hope these bring a smile!

## Part 2: Adding a tool

In [ ]:
# pushover_user = os.getenv("PUSHOVER_USER")
# pushover_token = os.getenv("PUSHOVER_TOKEN")
# pushover_url = "https://api.pushover.net/1/messages.json"

# if pushover_user:
#     if pushover_user.startswith("u"):
#         print("Pushover user found and looks good")
#     else:
#         print("Pushover user found but doesn't start with u")
# else:
#     print("Pushover user not found")

# if pushover_token:
#     if pushover_token.startswith("a"):
#         print("Pushover token found and looks good")
#     else:
#         print("Pushover token found but doesn't start with a")
# else:
#     print("Pushover token not found")

In [ ]:
# # Remember this?

# def push(message):
#     print(f"Push: {message}")
#     payload = {"user": pushover_user, "token": pushover_token, "message": message}
#     requests.post(pushover_url, data=payload)

In [ ]:
# # Now this:

# @function_tool
# def push_tool(message: str) -> str:
#     """ Send the given message to the user as a push notification """
#     payload = {"user": pushover_user, "token": pushover_token, "message": message}
#     result = requests.post(pushover_url, data=payload).status_code
#     return f"Push sent with API status code {result}"

In [15]:
Topic = "ross919gain370"
URL = f"https://ntfy.sh/{Topic}"

@function_tool
def Push(Message: str) -> str:
    """ Send the given message to the user as a push notification"""
    print (f"Push: {Message}")
    Result = requests.post(URL, data=Message.encode(encoding="utf-8")).status_code
    return f"Push sent with API code {Result}"

In [16]:
Push

FunctionTool(name='Push', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'Message': {'title': 'Message', 'type': 'string'}}, 'required': ['Message'], 'title': 'Push_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x000002A85C005E20>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [17]:
Push.description

'Send the given message to the user as a push notification'

In [19]:
Notifier = Agent(name="Notifier", model="gpt-4o-mini", instructions="You notify the user upon request",
                      tools=[Push])
with trace("Icecream has arrived"):
    Result = await Runner.run(Notifier, "Notify the user that the icecream is here")

print (Result.final_output)

Push: The ice cream is here!
The user has been notified that the ice cream is here!


## Now go and look at the trace

https://platform.openai.com/traces

## Part 3: Sessions (memory)

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [20]:
agent = Agent(name="Assistant", model="gpt-4o-mini")
Response = await Runner.run(agent, "Hi there, my name is Siddharth")
print (Response.final_output)
Response = await Runner.run(agent, "What is my name?")
print (Response.final_output)

Hi Siddharth! How can I assist you today?
I don’t have access to personal data unless you share it with me. What’s your name?


## Memory approach 1 - just manually pass in the list of dicts

In [29]:
agent = Agent(name="Assistant", model="gpt-4o-mini")
Response = await Runner.run(agent, "Hi there, my name is Siddharth")
print (Response.final_output)

NextInput = Response.to_input_list() + [{"role":"user","content":"What is my name?"}]
Response = await Runner.run(agent, NextInput)
print (Response.final_output)

Hi Siddharth! How can I assist you today?
Your name is Siddharth! How can I help you today?


## Another approach - use OpenAI Agents SDK built in SQLLite session

In [30]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

Session = SQLiteSession("12678")
Response = await Runner.run(agent, "Hi there, my name is Siddharth", session=Session)
print (Response.final_output)
Response = await Runner.run(agent, "What is my name?", session=Session)
print (Response.final_output)

Hi Siddharth! How can I assist you today?
Your name is Siddharth. How can I help you today?


# WOW

Can you believe how much we got done in Lab 1?!

Agents, Runner (Agent Loop), traces (Observability), Streaming, Function Tools, Memory!

Remember to check out the docs:  
https://openai.github.io/openai-agents-python/

Even better news: many of the lightweight Agent Frameworks are very similar, so you practically know them all..


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Make one of the Week 1 projects using OpenAI Agents SDK - like the digital twin or the Checklist loop. You will be astonished how easy it is.
            </span>
        </td>
    </tr>
</table>